In [2]:
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.table import Table
from astroquery.mast import Catalogs
import numpy as np

## Data loads

### Folder parsing

In [3]:
data_folder = '/Users/philvanlane/Documents/lc_ae/data/'

### Get TICs for all other targets

In [ ]:
# 2MASS   -> TWOMASS
# gaia2 -> GAIA
# HIP   -> HIP
# TYC --> TYC
# UCAC3 --> UCAC
# UCAC4 --> UCAC

In [8]:
targets = pd.read_csv(data_folder + 'mocadb_additional_old_FGKM.csv',dtype={'GaiaDR3_ID': str})

In [9]:
targets

,moca_oid,GaiaDR3_ID,twomass_id,moca_aid,moca_mtid,designation,ra,dec,spectral_type,banyan_prob,gmag,rmag,bmag,parallax_mas,prot_days,age_string_myr,age_myr,age_myr_unc,age_myr_unc_pos,age_myr_unc_neg
0,588165,414514646721724032,00520641+4937242,ALES1,HM,Gaia DR2 414514646721724032,13.026801,49.623368,(F0.7),99.2742,11.65513,11.36221,11.82684,1.333110,0.539814,1513.6,1513.6,NaN,NaN,NaN
1,588162,414508878584171904,00523184+4927067,ALES1,HM,Gaia DR2 414508878584171904,13.132742,49.451846,(F0.9),99.8956,12.32813,12.02769,12.52695,1.424150,NaN,1513.6,1513.6,NaN,NaN,NaN
2,587457,3446881477482884352,05195385+3035095,BER17,HM,Gaia DR2 3446881477482884352,79.974408,30.585998,(K9.2),99.7460,14.87300,13.91500,15.81800,0.339177,NaN,8128.3,8128.3,NaN,NaN,NaN
3,587449,3446833614367652992,05202905+3032414,BER17,HM,Gaia DR2 3446833614367652992,80.121103,30.544850,(K9.6),99.2981,15.29800,14.32300,16.19000,0.297538,NaN,8128.3,8128.3,NaN,NaN,NaN
4,584927,3157241252544842496,06574255+0816369,BER31,HM,Gaia DR2 3157241252544842496,104.427377,8.276973,(G3.1),99.7835,17.84200,17.35000,18.14000,0.014540,NaN,2951.2,2951.2,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3422,588317,4161178691096753792,18194336-0532161,UPK27,HM,Gaia DR2 4161178691096753792,274.930722,-5.537819,(M2.9),99.5378,9.87900,9.81700,10.98200,1.057900,NaN,1659.6,1659.6,NaN,NaN,NaN
3423,580506,1814376090584994048,20414930+2026343,UPK84,HM,Gaia DR2 1814376090584994048,310.455424,20.442829,(F0.4),99.7982,10.72600,10.44100,10.89600,1.057960,NaN,1023.3,1023.3,NaN,NaN,NaN
3424,580510,1817350888013852416,20400671+2018395,UPK84,HM,Gaia DR2 1817350888013852416,310.027918,20.310947,(F0.6),97.8228,12.63900,12.34700,12.81900,1.039400,NaN,1023.3,1023.3,NaN,NaN,NaN
3425,580503,1814348770293332224,20413149+2015036,UPK84,HM,Gaia DR2 1814348770293332224,310.381203,20.250988,(F1.7),99.3563,11.98300,11.66200,12.18100,1.029820,NaN,1023.3,1023.3,NaN,NaN,NaN


In [17]:
# targets['Gaia_ID'] = None
targets['TIC_ID'] = None
targets['msg'] = None

In [10]:
target = targets.iloc[65]

In [12]:
target

moca_oid                                 584620
GaiaDR3_ID                  3129900315375364608
twomass_id                     06575634+0627267
moca_aid                                  BER32
moca_mtid                                    HM
designation        Gaia DR2 3129900315375364608
ra                                   104.484711
dec                                    6.457441
spectral_type                            (G6.2)
banyan_prob                             98.8289
gmag                                       16.6
rmag                                     16.064
bmag                                     16.958
parallax_mas                           0.313665
prot_days                                   NaN
age_string_myr                           5495.4
age_myr                                  5495.4
age_myr_unc                                 NaN
age_myr_unc_pos                             NaN
age_myr_unc_neg                             NaN
Name: 65, dtype: object

In [20]:
for i in range(len(targets)):
    if i % 100 == 0:
        print(f'Processing target {i+1}/{len(targets)}')
    target = targets.iloc[i]

    if target.msg == '1 result' or target.msg == '2 results' or target.msg == '0 results':
        continue

    try:
        # if target.source_survey_name == 'gaia2':
        #     res = Catalogs.query_criteria(catalog='Tic',GAIA=target.source_survey_id)
        # elif target.source_survey_name == '2MASS':
        #     res = Catalogs.query_criteria(catalog='Tic',TWOMASS=target.source_survey_id)
        # elif target.source_survey_name == 'HIP':
        #     res = Catalogs.query_criteria(catalog='Tic',HIP=target.source_survey_id)
        # elif target.source_survey_name == 'TYC':
        #     res = Catalogs.query_criteria(catalog='Tic',TYC=target.source_survey_id)
        # elif target.source_survey_name == 'UCAC3':
        #     res = Catalogs.query_criteria(catalog='Tic',UCAC=target.source_survey_id)
        # elif target.source_survey_name == 'UCAC4':
        #     res = Catalogs.query_criteria(catalog='Tic',UCAC=target.source_survey_id)
        res = Catalogs.query_criteria(catalog='Tic',TWOMASS=target.twomass_id)
        
        if len(res) == 1:
            msg = '1 result'
            TIC_ID = res[0]['ID']
            # Gaia_ID = res[0]['GAIA']
        elif len(res) == 0:
            msg = '0 results'
            TIC_ID = 'N/A'
            # Gaia_ID = 'N/A'
        else:
            msg = f'{len(res)} results'
            TIC_ID = res[0]['ID']
            # Gaia_ID = res[0]['GAIA']
    except Exception as e:
        print(e)
        msg = "Error querying TIC"
        TIC_ID = 'N/A'
        # Gaia_ID = 'N/A'

    targets.loc[i,'TIC_ID'] = TIC_ID
    # targets.loc[i,'Gaia_ID'] = Gaia_ID
    targets.loc[i,'msg'] = msg

Processing target 1/3427
Processing target 101/3427
Processing target 201/3427
Processing target 301/3427
Processing target 401/3427
Processing target 501/3427
Processing target 601/3427
Processing target 701/3427
Processing target 801/3427
Processing target 901/3427
Processing target 1001/3427
Processing target 1101/3427
Processing target 1201/3427
Processing target 1301/3427
Processing target 1401/3427
Processing target 1501/3427
Processing target 1601/3427
Processing target 1701/3427
Processing target 1801/3427
Processing target 1901/3427
Processing target 2001/3427


Processing target 2101/3427


Processing target 2201/3427
Processing target 2301/3427


Processing target 2401/3427


Processing target 2501/3427


...

Processing target 3001/3427


Processing target 3101/3427


Processing target 3201/3427


Processing target 3301/3427


Processing target 3401/3427


In [21]:
targets

,moca_oid,GaiaDR3_ID,twomass_id,moca_aid,moca_mtid,designation,ra,dec,spectral_type,banyan_prob,...,bmag,parallax_mas,prot_days,age_string_myr,age_myr,age_myr_unc,age_myr_unc_pos,age_myr_unc_neg,TIC_ID,msg
0,588165,414514646721724032,00520641+4937242,ALES1,HM,Gaia DR2 414514646721724032,13.026801,49.623368,(F0.7),99.2742,...,11.82684,1.333110,0.539814,1513.6,1513.6,NaN,NaN,NaN,305831074,1 result
1,588162,414508878584171904,00523184+4927067,ALES1,HM,Gaia DR2 414508878584171904,13.132742,49.451846,(F0.9),99.8956,...,12.52695,1.424150,NaN,1513.6,1513.6,NaN,NaN,NaN,305825957,1 result
2,587457,3446881477482884352,05195385+3035095,BER17,HM,Gaia DR2 3446881477482884352,79.974408,30.585998,(K9.2),99.7460,...,15.81800,0.339177,NaN,8128.3,8128.3,NaN,NaN,NaN,2344582,1 result
3,587449,3446833614367652992,05202905+3032414,BER17,HM,Gaia DR2 3446833614367652992,80.121103,30.544850,(K9.6),99.2981,...,16.19000,0.297538,NaN,8128.3,8128.3,NaN,NaN,NaN,2512149,1 result
4,584927,3157241252544842496,06574255+0816369,BER31,HM,Gaia DR2 3157241252544842496,104.427377,8.276973,(G3.1),99.7835,...,18.14000,0.014540,NaN,2951.2,2951.2,NaN,NaN,NaN,235517244,1 result
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3422,588317,4161178691096753792,18194336-0532161,UPK27,HM,Gaia DR2 4161178691096753792,274.930722,-5.537819,(M2.9),99.5378,...,10.98200,1.057900,NaN,1659.6,1659.6,NaN,NaN,NaN,168020444,1 result
3423,580506,1814376090584994048,20414930+2026343,UPK84,HM,Gaia DR2 1814376090584994048,310.455424,20.442829,(F0.4),99.7982,...,10.89600,1.057960,NaN,1023.3,1023.3,NaN,NaN,NaN,331092487,1 result
3424,580510,1817350888013852416,20400671+2018395,UPK84,HM,Gaia DR2 1817350888013852416,310.027918,20.310947,(F0.6),97.8228,...,12.81900,1.039400,NaN,1023.3,1023.3,NaN,NaN,NaN,243577886,1 result
3425,580503,1814348770293332224,20413149+2015036,UPK84,HM,Gaia DR2 1814348770293332224,310.381203,20.250988,(F1.7),99.3563,...,12.18100,1.029820,NaN,1023.3,1023.3,NaN,NaN,NaN,331080243,1 result


In [22]:
targets.to_csv(data_folder + 'mocadb_add_TICS.csv', index=False)

## Deal with multiple TICs per target

In [4]:
targets_pt2 = pd.read_csv(data_folder + 'extra_target_TICS_2.csv')
targets_moca = pd.read_csv(data_folder + 'extra_target_TICS_mocadb.csv')

In [12]:
np.unique(targets_moca['msg'].values)

array(['1 result'], dtype=object)

In [23]:
targets

,moca_oid,GaiaDR3_ID,twomass_id,moca_aid,moca_mtid,designation,ra,dec,spectral_type,banyan_prob,...,bmag,parallax_mas,prot_days,age_string_myr,age_myr,age_myr_unc,age_myr_unc_pos,age_myr_unc_neg,TIC_ID,msg
0,588165,414514646721724032,00520641+4937242,ALES1,HM,Gaia DR2 414514646721724032,13.026801,49.623368,(F0.7),99.2742,...,11.82684,1.333110,0.539814,1513.6,1513.6,NaN,NaN,NaN,305831074,1 result
1,588162,414508878584171904,00523184+4927067,ALES1,HM,Gaia DR2 414508878584171904,13.132742,49.451846,(F0.9),99.8956,...,12.52695,1.424150,NaN,1513.6,1513.6,NaN,NaN,NaN,305825957,1 result
2,587457,3446881477482884352,05195385+3035095,BER17,HM,Gaia DR2 3446881477482884352,79.974408,30.585998,(K9.2),99.7460,...,15.81800,0.339177,NaN,8128.3,8128.3,NaN,NaN,NaN,2344582,1 result
3,587449,3446833614367652992,05202905+3032414,BER17,HM,Gaia DR2 3446833614367652992,80.121103,30.544850,(K9.6),99.2981,...,16.19000,0.297538,NaN,8128.3,8128.3,NaN,NaN,NaN,2512149,1 result
4,584927,3157241252544842496,06574255+0816369,BER31,HM,Gaia DR2 3157241252544842496,104.427377,8.276973,(G3.1),99.7835,...,18.14000,0.014540,NaN,2951.2,2951.2,NaN,NaN,NaN,235517244,1 result
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3422,588317,4161178691096753792,18194336-0532161,UPK27,HM,Gaia DR2 4161178691096753792,274.930722,-5.537819,(M2.9),99.5378,...,10.98200,1.057900,NaN,1659.6,1659.6,NaN,NaN,NaN,168020444,1 result
3423,580506,1814376090584994048,20414930+2026343,UPK84,HM,Gaia DR2 1814376090584994048,310.455424,20.442829,(F0.4),99.7982,...,10.89600,1.057960,NaN,1023.3,1023.3,NaN,NaN,NaN,331092487,1 result
3424,580510,1817350888013852416,20400671+2018395,UPK84,HM,Gaia DR2 1817350888013852416,310.027918,20.310947,(F0.6),97.8228,...,12.81900,1.039400,NaN,1023.3,1023.3,NaN,NaN,NaN,243577886,1 result
3425,580503,1814348770293332224,20413149+2015036,UPK84,HM,Gaia DR2 1814348770293332224,310.381203,20.250988,(F1.7),99.3563,...,12.18100,1.029820,NaN,1023.3,1023.3,NaN,NaN,NaN,331080243,1 result


In [ ]:
targets_multi = targets.loc[targets_pt2['msg'].isin(['2 results', '3 results'])]

In [16]:
for i,r in targets_multi.iterrows():
    
    try:
        if r.source_survey_name == 'gaia2':
            res = Catalogs.query_criteria(catalog='Tic',GAIA=r.source_survey_id)
        elif r.source_survey_name == '2MASS':
            res = Catalogs.query_criteria(catalog='Tic',TWOMASS=r.source_survey_id)
        elif r.source_survey_name == 'HIP':
            res = Catalogs.query_criteria(catalog='Tic',HIP=r.source_survey_id)
        elif r.source_survey_name == 'TYC':
            res = Catalogs.query_criteria(catalog='Tic',TYC=r.source_survey_id)
        elif r.source_survey_name == 'UCAC3':
            res = Catalogs.query_criteria(catalog='Tic',UCAC=r.source_survey_id)
        elif r.source_survey_name == 'UCAC4':
            res = Catalogs.query_criteria(catalog='Tic',UCAC=r.source_survey_id)
        
        for r_row in res:
            print(r.object,',',r.source_survey_name,',',r.source_survey_id,',',r.ref,',', r_row['ID'],',',r_row['GAIA'])
    except Exception as e:
        print("Error querying TIC")

01005643-0426561 , 2MASS , 01005643-0426561 , Newton2018 , 398570754 , 2525276565243086080
01005643-0426561 , 2MASS , 01005643-0426561 , Newton2018 , 471012716 , --
01045368-1807292 , 2MASS , 01045368-1807292 , Newton2018 , 471016687 , --
01045368-1807292 , 2MASS , 01045368-1807292 , Newton2018 , 268633473 , 2358022227591374336
02042754-0152560 , 2MASS , 02042754-0152560 , Newton2018 , 471012310 , --
02042754-0152560 , 2MASS , 02042754-0152560 , Newton2018 , 250411375 , 2506306454521815040
02130805-1901526 , 2MASS , 02130805-1901526 , Newton2018 , 471016691 , --
02130805-1901526 , 2MASS , 02130805-1901526 , Newton2018 , 268803844 , 5143185923500965504
02141251-0357434 , 2MASS , 02141251-0357434 , Newton2018 , 471012790 , --
02141251-0357434 , 2MASS , 02141251-0357434 , Newton2018 , 332819239 , 2492866780297999232
03210557-0526345 , 2MASS , 03210557-0526345 , Newton2018 , 471012778 , --
03210557-0526345 , 2MASS , 03210557-0526345 , Newton2018 , 279073853 , 5170070254110129792
03353849-0

In [99]:
targets.to_csv(data_folder + 'extra_target_TICS.csv', index=False)

In [22]:
res = Catalogs.query_criteria(catalog='Tic',ID=398570754)

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [21]:
targets_multi

,object,source_survey_name,source_survey_id,TIC_ID,ref,Gaia_ID,msg
16,01005643-0426561,2MASS,01005643-0426561,398570754.0,Newton2018,2525276565243086080,2 results
20,01045368-1807292,2MASS,01045368-1807292,471016687.0,Newton2018,--,2 results
53,02042754-0152560,2MASS,02042754-0152560,250411375.0,Newton2018,2506306454521815040,2 results
56,02130805-1901526,2MASS,02130805-1901526,268803844.0,Newton2018,5143185923500965504,2 results
58,02141251-0357434,2MASS,02141251-0357434,471012790.0,Newton2018,--,2 results
...,...,...,...,...,...,...,...
3424,J13224492+6441334,2MASS,13224492+6441334,289521781.0,Newton2016,1677768333540094848,2 results
3504,J14085027+7316300,2MASS,14085027+7316300,198512076.0,Newton2016,1699503788635584896,2 results
3508,J14095169+0724258,2MASS,14095169+0724258,471012653.0,Newton2016,3674243817630610304,2 results
3848,J15460441+0441311,2MASS,15460441+0441311,272519426.0,Newton2016,4426270940229616256,2 results


In [95]:
targets.loc[targets['msg'] == 'Error querying TIC']

,object,source_survey_name,source_survey_id,ref,TIC_ID,Gaia_ID,msg


In [35]:
r = Catalogs.query_criteria(catalog='Tic',HIP='17695')

In [36]:
len(r)

1

In [37]:
r

ID,version,HIP,TYC,UCAC,TWOMASS,SDSS,ALLWISE,GAIA,APASS,KIC,objType,typeSrc,ra,dec,POSflag,pmRA,e_pmRA,pmDEC,e_pmDEC,PMflag,plx,e_plx,PARflag,gallong,gallat,eclong,eclat,Bmag,e_Bmag,Vmag,e_Vmag,umag,e_umag,gmag,e_gmag,rmag,e_rmag,imag,e_imag,zmag,e_zmag,Jmag,e_Jmag,Hmag,e_Hmag,Kmag,e_Kmag,TWOMflag,prox,w1mag,e_w1mag,w2mag,e_w2mag,w3mag,e_w3mag,w4mag,e_w4mag,GAIAmag,e_GAIAmag,Tmag,e_Tmag,TESSflag,SPFlag,Teff,e_Teff,logg,e_logg,MH,e_MH,rad,e_rad,mass,e_mass,rho,e_rho,lumclass,lum,e_lum,d,e_d,ebv,e_ebv,numcont,contratio,disposition,duplicate_id,priority,eneg_EBV,epos_EBV,EBVflag,eneg_Mass,epos_Mass,eneg_Rad,epos_Rad,eneg_rho,epos_rho,eneg_logg,epos_logg,eneg_lum,epos_lum,eneg_dist,epos_dist,distflag,eneg_Teff,epos_Teff,TeffFlag,gaiabp,e_gaiabp,gaiarp,e_gaiarp,gaiaqflag,starchareFlag,VmagFlag,BmagFlag,splists,e_RA,e_Dec,RA_orig,Dec_orig,e_RA_orig,e_Dec_orig,raddflag,wdflag,objID
str9,str8,int64,str1,str10,str16,str1,str1,str19,str6,int64,str4,str7,float64,float64,str7,float64,float64,float64,float64,str5,float64,float64,str5,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str19,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str5,str5,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str5,float64,float64,float64,float64,float64,float64,int64,float64,str1,str1,float64,float64,float64,str1,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str6,float64,float64,str5,float64,float64,float64,float64,int64,str1,str5,str8,str13,float64,float64,float64,float64,float64,float64,int64,int64,str9
333680372,20190415,17695,--,441-005220,03472333-0158195,--,--,3250328209054347264,743879,--,STAR,tmgaia2,56.8472550051883,-1.97220704058793,tmgaia2,180.673,0.0974111,-274.184,0.0900251,gaia2,59.5161,0.0564864,gaia2,189.748455191891,-40.9688753713107,54.0624188275218,-21.3708339108133,13.066,0.023,11.559,0.066,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,7.804,0.026,7.174,0.051,6.933,0.024,AAA-111-111-000-0-0,nan,nan,nan,nan,nan,nan,nan,nan,nan,10.4962,0.001207,9.3168,0.00742333,cdwrf,cdwrf,3439.0,157.0,4.76033,0.00727509,nan,nan,0.473917,0.014208,0.471696,0.020377,4.43155,0.207294,DWARF,0.0283027086,0.00690207444,16.794,0.01605,0.0,0.0,37,0.000232819366,--,--,0.009350562,0.0,0.0,--,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,bj2018,nan,nan,cdwrf,11.8615,0.003778,9.37494,0.001832,1,--,cdwrf,apassdr9,cooldwarfs_v8,1.51075293261413,1.39577414218954,56.8480333639031,-1.9733875552237,0.044548841177301,0.032784983529585,1,0,522811716


### Data Load

In [23]:
m_m_2022 = data_folder + 'apjac7da8t2_ascii.txt'
M_WD_2022 = data_folder + 'apjac7da8t3_ascii.txt'
readme_2023 = data_folder + '2023_ReadMe.txt'
t1_2023 = data_folder + '2023_table1.dat'

M_M_formatted = data_folder + 'Pass2022_MM_binaries.csv'
M_WD_formatted = data_folder + 'Pass2022_M_WD_binaries.csv'
t1_formatted = data_folder + 'Pass2023_table1.csv'

In [28]:
f = data_folder + '2024_apjad3631t3_mrt.txt'
t = Table.read(f, format="ascii.cds")

In [30]:
df = t.to_pandas()

In [31]:
df.to_csv(data_folder + 'Pass2024_MM_systems.csv', index=False)

In [13]:
meta_headers = pd.read_csv(readme_2023,
                               skiprows=92,
                               nrows=1,
                               header=None,
                               delim_whitespace=True
                         ).values[0]

/var/folders/lf/sq3cxxf17kv47p5lcv6w5t5r0000gn/T/ipykernel_50749/3788376468.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  meta_headers = pd.read_csv(readme_2023,


In [14]:
meta_headers

array(['Bytes', 'Format', 'Units', 'Label', 'Explanations'], dtype=object)

In [19]:
meta_data = pd.read_fwf(readme_2023,
                               widths=[9,6,7,12,100],
                               header=88,
                                nrows=19,
                               names=meta_headers
                         )

In [20]:
meta_data = meta_data.dropna(subset=['Bytes'])

In [21]:
meta_data

,Bytes,Format,Units,Label,Explanations
0,1- 13,A13,---,Name,Star name
1,15- 37,A23,---,2MASS,2MASS identifier
2,39- 39,A1,---,Inst,"Instrument, T:TRES; C:CHIRON"
3,41- 42,I2,---,Nobs,[2/21] Number of spectroscopic observations
4,44- 48,F5.3,Msun,Mass,"[0.1/0.29] Stellar mass, solar units"
5,50- 54,F5.3,Rsun,Rad,"[0.15/0.3] Stellar radius, solar units"
6,56- 61,F6.2,0.1nm,EWHa,[-17.2/-1.02] Median equivalent width of
8,63- 66,F4.1,km/s,vsini,"[0/56.3] Median vsini, projected rotational"
10,68- 71,F4.1,km/s,e_vsini,[0/11] vsini standard deviation
11,73- 76,F4.2,---,sini,[0.29/1.0]? sin(inclination)


In [22]:
data_widths = []

for k in range(len(meta_data['Bytes'].values)):
    c = meta_data['Bytes'].values[k]
    try:
        i = c.index('-')
        start = int(c[0:i])
        end = int(c[i+1:])
        data_widths.append(end - start + 2)
    except:
        data_widths.append(2)

In [24]:
data = pd.read_fwf(t1_2023,
                               widths=data_widths,
                               header=None,
                               names=meta_data['Label'].values
                         )

In [25]:
data

,Name,2MASS,Inst,Nobs,Mass,Rad,EWHa,vsini,e_vsini,sini,MedeRVel,logPChi2,Per,Amp,Ref
0,2MA0505-4756,2MASS J05051443-4756154,C,4,0.159,0.195,-2.37,13.9,0.3,1.00,0.317,-0.04,0.726,0.004,3.0
1,2MA2009-0113,2MASS J20091824-0113377,T,4,0.146,0.186,-6.15,4.5,0.5,0.66,0.047,-0.95,1.374,0.017,2.0
2,APCOL,2MASS J06045215-3433360,C,4,0.279,0.285,-8.26,15.7,0.4,1.00,0.131,-0.46,1.013,0.009,3.0
3,APM0237-5928,2MASS J02363244-5928057,C,4,0.147,0.187,-5.69,20.5,0.3,1.00,0.213,-1.76,0.469,0.025,2.0
4,G141-36,2MASS J18481752+0741210,T,12,0.136,0.179,-3.89,3.2,0.4,NaN,0.027,-0.29,2.756,0.005,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,WIS1540-5101,2MASS J15404341-5101357,C,4,0.105,0.158,-1.21,0.6,0.3,NaN,0.020,-0.87,93.702,0.006,2.0
119,WIS1824-0536,2MASS J18242359-0536488,T,2,0.115,0.165,-7.38,3.4,2.1,0.57,0.126,-1.46,1.422,0.007,9.0
120,WT2458,2MASS J09455843-3253299,C,4,0.186,0.214,-5.20,20.5,0.6,0.59,0.222,-0.20,0.312,0.009,3.0
121,WT84,2MASS J02172845-5922435,C,4,0.126,0.172,-10.48,4.1,0.5,NaN,0.086,-0.28,5.232,0.018,2.0


In [26]:
data.to_csv(t1_formatted, index=False)

In [40]:
ra_hours = data['RAh'].values  # RA hours
ra_minutes = data['RAm'].values  # RA minutes
ra_seconds = data['RAs'].values  # RA seconds

dec_degrees = data['DEd'].values  # Dec degrees
dec_minutes = data['DEm'].values  # Dec minutes
dec_seconds = data['DEs'].values  # Dec seconds

# Create SkyCoord object
coord = SkyCoord(ra=ra_hours*u.hour + ra_minutes*u.minute + ra_seconds*u.second,
                 dec=dec_degrees*u.deg + dec_minutes*u.arcmin + dec_seconds*u.arcsec, 
                 frame='icrs')

# Convert RA and Dec to degrees
ra_in_degrees = coord.ra.deg
dec_in_degrees = coord.dec.deg

data['RA_deg'] = ra_in_degrees
data['DE_deg'] = dec_in_degrees

In [38]:
data.to_csv(datafile_formatted)